# Live feature extraction — logreg_v4

This notebook walks through **exactly** how the bot computes the 18 features that feed `logreg_v4` during live trading. We replay a real slot from `data/btc_live_1s.csv` + `data/live_orderbook_snapshots.csv`, build the snapshot dict the way `LogRegEdgeStrategy._predict()` does at `src/strategies/logreg_edge.py:274-283`, and then invoke each numpy helper exactly as `LogRegV4Model.predict()` does at `src/models/logreg_v4_model.py:126-243`.

At the end we cross-check our manually-assembled feature vector against the one that `LogRegV4Model.predict()` produces — they should match to the last digit.

## The 18 features (from `models/logreg_v4/logreg_meta.json`)

| Group | Features |
|---|---|
| Time | `time_to_expiry` |
| Momentum | `ret_5s`, `ret_15s`, `ret_30s`, `ret_60s`, `ret_180s` |
| Volatility | `vol_15s`, `vol_30s`, `vol_60s`, `vol_ratio_15_60` |
| Volume | `volume_surge_ratio`, `vwap_deviation`, `cvd_60s` |
| Indicators | `rsi_14`, `td_setup_net` |
| Orderbook | `spread`, `ob_imbalance`, `ob_cross_imbalance` |

The only inputs the live code has access to are:
- `btc_prices`: list of `(timestamp, mid_price, tick_count)` for last 300s from `BtcPriceFeed.get_recent_prices()`
- `yes_book` / `no_book`: current `OrderBook` snapshots (top levels)
- `slot_expiry_ts`, `now_ts`: slot timing

Everything else is derived numpy on the BTC price array + direct reads off the top-3 depth of the orderbook.

## 1. Setup

In [ ]:
import os, sys, json, pickle
import numpy as np
import pandas as pd

REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, REPO)

# Reuse the exact numpy helpers the live model uses. Train/serve parity:
# every cell below calls these same functions, so we're not reimplementing
# anything here — we're just narrating the pipeline.
from src.models.logreg_v4_model import (
    LogRegV4Model,
    _asof, _ret, _vol, _rsi, _td_setup_net,
    _volume_surge, _vwap_deviation, _cvd,
)
from src.api.types import OrderBook, OrderBookEntry

V4_DIR = os.path.join(REPO, "models/logreg_v4")
meta = json.load(open(os.path.join(V4_DIR, "logreg_meta.json")))
print(f"Model version : {meta['model_version']}")
print(f"n_features    : {len(meta['features'])}")
print(f"Calibrated    : {meta.get('calibrated', False)}")
print("Features      :", meta["features"])


## 2. Replay one real slot from disk

We need the live inputs: (a) a 300s rolling BTC price window and (b) a YES/NO orderbook snapshot. We don't have a WebSocket here, so we reconstruct them from the saved CSVs. For feature computation, this is indistinguishable from the live path — the helpers only see numpy arrays.

In [ ]:
btc = pd.read_csv(os.path.join(REPO, "data/btc_live_1s.csv"))
ob  = pd.read_csv(os.path.join(REPO, "data/live_orderbook_snapshots.csv"))
print(f"BTC rows : {len(btc):,}")
print(f"OB rows  : {len(ob):,}  ({ob['slot_ts'].nunique()} slots)")
btc.head(2)

In [ ]:
# Pick a slot that has plenty of orderbook snapshots so we can land at a
# decision time with 200s+ of BTC history behind us.
slot_ts = int(ob['slot_ts'].value_counts().idxmax())
elapsed = 150  # seconds into the slot — simulates 'bot woke up at t=slot+150s'
now_ts  = float(slot_ts + elapsed)
slot_expiry_ts = float(slot_ts + 300)

print(f"slot_ts        : {slot_ts}")
print(f"elapsed        : {elapsed}s")
print(f"now_ts         : {now_ts}")
print(f"slot_expiry_ts : {slot_expiry_ts}")
print(f"tte            : {slot_expiry_ts - now_ts}s")

### 2a. Build `btc_prices` as a 300s rolling window

In live trading, `BtcPriceFeed.get_recent_prices(300)` returns a list of `(ts, mid, tick_count)` for the last 300 seconds. We slice the CSV to the same shape.

In [ ]:
window = btc[(btc['timestamp'] >= now_ts - 300) & (btc['timestamp'] <= now_ts)].copy()
# Emulate the 3-tuple the live feed produces: (ts, mid, tick_count). The
# CSV has 'close' and 'volume' per 1s bar, which is what the live feed
# also accumulates.
btc_prices = list(zip(
    window['timestamp'].astype(float),
    window['close'].astype(float),
    window['volume'].astype(float),
))
print(f"btc_prices rows : {len(btc_prices)}")
print(f"first          : {btc_prices[0]}")
print(f"last           : {btc_prices[-1]}")

### 2b. Build `yes_book` / `no_book` from the orderbook snapshot CSV

The CSV only keeps aggregated top-3 depth per side (`bid_depth_3`, `ask_depth_3`), not the full ladder. That's fine — the live code only consumes `bids[:3]` / `asks[:3]` in `logreg_v4_model.py:175-176`, so we can fake a single synthetic level carrying the aggregated depth and the result is identical.

In [ ]:
up_rows = ob[(ob['slot_ts'] == slot_ts) & (ob['side'] == 'up')].reset_index(drop=True)
dn_rows = ob[(ob['slot_ts'] == slot_ts) & (ob['side'] == 'down')].reset_index(drop=True)
up_row = up_rows.iloc[(up_rows['elapsed_s'] - elapsed).abs().idxmin()]
dn_row = dn_rows.iloc[(dn_rows['elapsed_s'] - elapsed).abs().idxmin()]

def fake_book(row, token_id):
    # Single-level book whose top-3 depth sum matches the CSV aggregate.
    return OrderBook(
        market_id=f'slot_{slot_ts}',
        token_id=token_id,
        asks=[OrderBookEntry(price=float(row['best_ask']), size=float(row['ask_depth_3']))],
        bids=[OrderBookEntry(price=float(row['best_bid']), size=float(row['bid_depth_3']))],
        tick_size=0.01,
    )

yes_book = fake_book(up_row, token_id='YES_TOKEN')
no_book  = fake_book(dn_row, token_id='NO_TOKEN')
print(f"YES: bid={yes_book.bids[0].price}({yes_book.bids[0].size:.1f})  "
      f"ask={yes_book.asks[0].price}({yes_book.asks[0].size:.1f})")
print(f"NO : bid={no_book.bids[0].price}({no_book.bids[0].size:.1f})  "
      f"ask={no_book.asks[0].price}({no_book.asks[0].size:.1f})")

### 2c. Assemble the snapshot dict

This is byte-for-byte the dict `LogRegEdgeStrategy._predict()` passes into `LogRegV4Model.predict()` — see `src/strategies/logreg_edge.py:274-283`.

In [ ]:
# Strike price: in live trading this comes from the market question.
# For replay, BTC price at slot start is the strike.
strike_price = float(btc[btc['timestamp'] <= slot_ts].iloc[-1]['close'])

snapshot = {
    'btc_prices': btc_prices,
    'yes_book': yes_book,
    'no_book': no_book,
    'yes_history': [],            # not used by v4
    'yes_ob_history': [],         # only v5 features need this
    'question': '',
    'strike_price': strike_price,
    'slot_expiry_ts': slot_expiry_ts,
    'now_ts': now_ts,
}
print(f"strike_price = {strike_price:.2f}")
print(f"snapshot keys = {list(snapshot.keys())}")

## 3. Compute the 18 features, one group at a time

The helpers all operate on two numpy arrays pulled out of `btc_prices`: a timestamp array `ts_arr` and a price array `px_arr`. Volume uses `vol_arr`. This is exactly how `LogRegV4Model.predict()` starts at lines 155-158.

In [ ]:
ts_arr  = np.array([float(p[0]) for p in btc_prices])
px_arr  = np.array([float(p[1]) for p in btc_prices])
vol_arr = np.array([float(p[2]) for p in btc_prices])
print(f"ts_arr  shape={ts_arr.shape}   range=[{ts_arr.min():.0f}, {ts_arr.max():.0f}]")
print(f"px_arr  shape={px_arr.shape}   range=[{px_arr.min():.2f}, {px_arr.max():.2f}]")
print(f"vol_arr shape={vol_arr.shape}  range=[{vol_arr.min():.3f}, {vol_arr.max():.3f}]")

### 3.1 `time_to_expiry`

Trivial: slot closes at `slot_expiry_ts`, we are at `now_ts`.

In [ ]:
tte = max(0.0, slot_expiry_ts - now_ts)
print(f"time_to_expiry = {tte:.1f}s")

### 3.2 Momentum — `ret_5s`, `ret_15s`, `ret_30s`, `ret_60s`, `ret_180s`

`_ret()` at `logreg_v4_model.py:304` is a pure percentage change over a lookback window:

$$\mathrm{ret}_{L} = \frac{P(\text{now}) - P(\text{now} - L)}{P(\text{now} - L)}$$

Both endpoints are looked up with `searchsorted` via `_asof()` — this is the same causal `as-of` join the training script uses, so there's no look-ahead bias between train and serve.

In [ ]:
# Worked example for ret_60s — show the two endpoints the helper picks.
ci = _asof(ts_arr, now_ts)
pi = _asof(ts_arr, now_ts - 60)
print(f"as-of now      : idx={ci}  ts={ts_arr[ci]:.0f}  px={px_arr[ci]:.2f}")
print(f"as-of now - 60 : idx={pi}  ts={ts_arr[pi]:.0f}  px={px_arr[pi]:.2f}")
manual_ret_60 = (px_arr[ci] - px_arr[pi]) / px_arr[pi]
helper_ret_60 = _ret(ts_arr, px_arr, now_ts, 60)
print(f"manual         : {manual_ret_60:+.6f}")
print(f"helper _ret    : {helper_ret_60:+.6f}")
assert abs(manual_ret_60 - helper_ret_60) < 1e-12

In [ ]:
rets = {f'ret_{L}s': _ret(ts_arr, px_arr, now_ts, L) for L in (5, 15, 30, 60, 180)}
for k, v in rets.items():
    print(f"  {k:<10} {v:+.6f}")

### 3.3 Volatility — `vol_15s`, `vol_30s`, `vol_60s`, `vol_ratio_15_60`

`_vol()` at `logreg_v4_model.py:312` is the sample std of log-returns inside a trailing window:

$$\mathrm{vol}_{L} = \operatorname{std}\!\left(\left\{\log\tfrac{P_{i+1}}{P_i} : t_i \in [\text{now}-L,\,\text{now}]\right\}\right)$$

It returns `0.0` if fewer than 3 price points are in the window (degenerate window guard).

`vol_ratio_15_60 = vol_15s / vol_60s` is a short-vs-long regime-change detector — spikes when recent vol exceeds trailing vol.

In [ ]:
v15 = _vol(ts_arr, px_arr, now_ts, 15)
v30 = _vol(ts_arr, px_arr, now_ts, 30)
v60 = _vol(ts_arr, px_arr, now_ts, 60)
vol_ratio = (v15 / v60) if v60 > 0 else 0.0
print(f"vol_15s         = {v15:.6f}")
print(f"vol_30s         = {v30:.6f}")
print(f"vol_60s         = {v60:.6f}")
print(f"vol_ratio_15_60 = {vol_ratio:.4f}")

### 3.4 Volume features — `volume_surge_ratio`, `vwap_deviation`, `cvd_60s`

- **`volume_surge_ratio`** = `mean(volume[-15s]) / mean(volume[-60s])`. >1 → recent tick intensity outpaces the trailing minute (surge). Guard returns 0 when denom is zero.
- **`vwap_deviation`** = `(spot - vwap_60s) / spot`, i.e. how far today's last print is from the 60s volume-weighted mean — positive means price is above VWAP (buyers leaning in).
- **`cvd_60s`** = normalized cumulative volume delta over 60s. For each tick in the window, tag its volume as buy if `Δprice ≥ 0` else sell, then return `(buy - sell) / (buy + sell)`. Signed imbalance in `[-1, +1]`.

In [ ]:
surge = _volume_surge(ts_arr, vol_arr, now_ts)
vwap_dev = _vwap_deviation(ts_arr, px_arr, vol_arr, now_ts)
cvd = _cvd(ts_arr, px_arr, vol_arr, now_ts)
print(f"volume_surge_ratio = {surge:.4f}")
print(f"vwap_deviation     = {vwap_dev:+.6f}")
print(f"cvd_60s            = {cvd:+.4f}")

### 3.5 Classic indicators — `rsi_14`, `td_setup_net`

- **`rsi_14`** — standard 14-period Wilder-ish RSI computed over the most recent 15 price points. Falls back to 50 (neutral) if the window doesn't have enough history.
- **`td_setup_net`** — simplified TD Sequential setup: over the last ~17 bars, count the number of consecutive bars where `close[i] < close[i-4]` (bullish setup) vs `close[i] > close[i-4]` (bearish setup), and return `bear_count - bull_count`. Positive value = bearish pressure mounting.

In [ ]:
rsi = _rsi(ts_arr, px_arr, now_ts)
td  = _td_setup_net(ts_arr, px_arr, now_ts)
print(f"rsi_14       = {rsi:.2f}")
print(f"td_setup_net = {td:+.1f}")

### 3.6 Orderbook features — `spread`, `ob_imbalance`, `ob_cross_imbalance`

These are the only features that touch the orderbook. They read top-of-book prices and the summed top-3 depth on each side.

- **`spread`** = `yes_ask - yes_bid` (absolute, in USDC).
- **`ob_imbalance`** = `(YES_bid_d3 - YES_ask_d3) / (YES_bid_d3 + YES_ask_d3)`. Signed top-3 imbalance on the YES side, in `[-1, +1]`.
- **`ob_cross_imbalance`** = `YES_bid_d3 / (YES_bid_d3 + NO_bid_d3)`. Cross-token demand: fraction of aggregate bid depth sitting on the YES book vs. the NO book. A pure market-demand proxy.

In [ ]:
yes_bid = float(yes_book.bids[0].price)
yes_ask = float(yes_book.asks[0].price)
yes_bid_d = sum(float(l.size) for l in yes_book.bids[:3])
yes_ask_d = sum(float(l.size) for l in yes_book.asks[:3])
no_bid_d  = sum(float(l.size) for l in no_book.bids[:3])

spread = max(0.0, yes_ask - yes_bid)
ob_total = yes_bid_d + yes_ask_d
ob_imbalance = (yes_bid_d - yes_ask_d) / ob_total if ob_total > 0 else 0.0
cross_total = yes_bid_d + no_bid_d
ob_cross_imbalance = yes_bid_d / cross_total if cross_total > 0 else 0.5

print(f"spread             = {spread:.4f}")
print(f"ob_imbalance       = {ob_imbalance:+.4f}")
print(f"ob_cross_imbalance = {ob_cross_imbalance:.4f}")

## 4. Assemble the full feature vector

`LogRegV4Model.predict()` builds an 18-element dict, then projects it onto `self.feature_names` (which comes from `logreg_meta.json`) before passing it through the scaler. This dict→vector projection is how the same live code handles v3/v4/v5 without changes — each checkpoint declares its own feature list in its meta file.

In [ ]:
feats = {
    'time_to_expiry': tte,
    'ret_5s':   rets['ret_5s'],
    'ret_15s':  rets['ret_15s'],
    'ret_30s':  rets['ret_30s'],
    'ret_60s':  rets['ret_60s'],
    'ret_180s': rets['ret_180s'],
    'vol_15s':  v15,
    'vol_30s':  v30,
    'vol_60s':  v60,
    'vol_ratio_15_60':    vol_ratio,
    'volume_surge_ratio': surge,
    'vwap_deviation':     vwap_dev,
    'cvd_60s':            cvd,
    'rsi_14':             rsi,
    'td_setup_net':       td,
    'spread':             spread,
    'ob_imbalance':       ob_imbalance,
    'ob_cross_imbalance': ob_cross_imbalance,
}
pd.Series(feats, name='value').to_frame()

In [ ]:
# Project onto the feature order the trained model/scaler expect.
feature_names = meta['features']
X = np.array([[feats[name] for name in feature_names]], dtype=float)
print(f"X.shape = {X.shape}")
print(X)

## 5. Scale → logistic regression → isotonic calibration

Three steps, all pure sklearn:
1. `scaler.transform(X)` — `StandardScaler` fit on training data (18 scalar means + stds).
2. `model.predict_proba(X_scaled)[0, 1]` — raw logistic regression probability.
3. `calibrator.predict(p_raw)` — post-hoc isotonic calibration (v4-only; v3 had no calibrator).

Step 3 is what separates v4's 0.207 Brier from v3's uncalibrated output — the isotonic map squashes the raw sigmoid into the empirical hit-rate curve on the validation set.

In [ ]:
scaler = pickle.load(open(os.path.join(V4_DIR, 'logreg_scaler.pkl'), 'rb'))
model  = pickle.load(open(os.path.join(V4_DIR, 'logreg_model.pkl'), 'rb'))
calibrator = pickle.load(open(os.path.join(V4_DIR, 'logreg_calibrator.pkl'), 'rb'))

Xs = scaler.transform(X)
prob_raw = float(model.predict_proba(Xs)[0, 1])
prob_cal = float(calibrator.predict(np.array([prob_raw]))[0])
print(f"scaled X (first 6): {Xs[0, :6].round(3)}...")
print(f"p_up (raw)  = {prob_raw:.4f}")
print(f"p_up (cal)  = {prob_cal:.4f}  ← this is what LogRegEdgeStrategy sees")

## 6. Cross-check against the production path

Load the same model through `LogRegV4Model.load()` and call `predict(snapshot)` on the exact same snapshot dict. The returned probability and the model's `last_features` dict should match what we computed by hand.

In [ ]:
loaded = LogRegV4Model.load(V4_DIR)
result = loaded.predict(snapshot)
print(f"model_version  : {result.model_version}")
print(f"feature_status : {result.feature_status}")
print(f"prob_yes       : {result.prob_yes:.4f}")

# Compare the two feature vectors element-by-element.
prod = loaded.last_features
diff = pd.DataFrame({
    'manual':     [feats[k] for k in feature_names],
    'production': [prod[k] for k in feature_names],
}, index=feature_names)
diff['Δ'] = (diff['manual'] - diff['production']).abs()
print(f"\nmax |Δ| = {diff['Δ'].max():.2e}")
diff

If `max |Δ|` above is essentially zero (< 1e-12) and `p_up` from the production path matches what we computed in section 5, the notebook has proven end-to-end parity: every feature shown here is the *exact* value the live model sees on this snapshot.

## What this notebook does not cover

- **v5 features** (`microprice_delta`, `imbalance_mean_10s`, `imbalance_slope_10s`, `bid_ask_size_ratio_change_5s`) — computed only when the loaded model declares them, and need a rolling `yes_ob_history` buffer that's populated in `LogRegEdgeStrategy.analyze()`. The v4 meta doesn't list them so they're skipped at `logreg_v4_model.py:216`.
- **Edge computation and Kelly sizing** — happens in `LogRegEdgeStrategy._check_entry()` after `predict()` returns; `p_up` is compared against `q_t + c_t + delta` there, not inside the model.
- **Isotonic calibration training** — this notebook loads a pre-fit calibrator; the fit itself happens in `scripts/train_logreg_v3.py` on the validation slice.